# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# cargar archivos
orders = pd.read_csv('rappiplus_orders_raw.csv') 
catalog = pd.read_csv('rappiplus_catalog.csv')
marketing = pd.read_csv('rappiplus_marketing_spend.csv')

In [3]:
# explorando datasets
orders.info()
orders.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


✍️ **Comentarios:** 

El dataset **orders** tiene 25,100 filas y 12 columnas. 

Hay valores faltantes en columnas pais (300 valores), dispositivo (20 valores), fuente_referencia (30 valores), nombre_producto (30 valores), categoria_producto (80 valores), cantidad (50 valores), precio_unitario (50 valores) y monto_descuento (50 valores). 

El dataset tiene 8 columnas tipo 'object' lo que significa que tienen tipos de datos string, como letras, numeros o simbolos y 4 tipo 'float' lo que indica que son numeros reales.

Las columnas tipo object son: id_pedido, id_usuario, fecha_hora_pedido, pais, dispositivo, fuente_referencia, nombre_producto, categoria_producto. Las columnas tipo float son: cantidad, precio_unitario, monto_descuento, monto_total.      


In [4]:
print('\nValores Faltantes (por columna):')
display(orders.isna().sum().sort_values(ascending=False).head(15))


Valores Faltantes (por columna):


pais                  300
categoria_producto     80
cantidad               50
precio_unitario        50
monto_descuento        50
fuente_referencia      30
nombre_producto        30
dispositivo            20
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
monto_total             0
dtype: int64

In [5]:
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


✍️ **Comentarios:** 

La tabla estadistica anterior describe los datos numericos mostrandos indicadores como cantidad de registros, media, desviacion estandar, valores minimos y maximos e interquartils. 

La columna `cantidad` tiene 25,050 registros, siendo el valor minimo -2.0 y el valor maximo 20,000. Ademas, tiene una desviacion estandar de 296.27 y la media es 7.09. Aparentemente no hay aparición de outliers ya que el interquatile Q1, Q2 y Q3 tienen una diferencia proporcional. Sin embargo, mas adelante se hare el respectivo analisis. 

La columna `precio unitario` tiene 25,050 registros, el valor minimo es 20.03 y el valor maximo es 499.96. Su desviacion estandar es de 138.73 y la media es 259.30.

In [6]:
# explorando datasets
catalog.info()
catalog.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


✍️ **Comentarios:** 

El dataset **catalog** tiene 7 filas y 4 columnas. 

No hay datos faltantes. 

El dataset tiene 3 columnas tipo 'object' lo que significa que tienen tipos de datos string, como letras, numeros o simbolos y 1 columna tipo 'float' lo que indica que son numeros reales.

Las columnas tipo object son: nombre_producto, categoria_producto y proveedor. La columna tipo float es costo_unitario.    


In [7]:
print('\nValores Faltantes (por columna):')
display(catalog.isna().sum().sort_values(ascending=False).head(15))


Valores Faltantes (por columna):


nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64

In [8]:
catalog.describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


✍️ **Comentarios:** 

La tabla estadistica anterior indica que: 

La columna `costo unitario` tiene 7 registros, siendo el valor minimo 10.12, el valor maximo es 286.68, la desviacion estandar es 111.01, la media es 102.25. Es necesario analizar la data ya que hay aparición de outliers. 


In [9]:
# explorar datasets
marketing.info()
marketing.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


✍️ **Comentarios:** 

El dataset **marketing** tiene 1620 filas y 5 columnas. 

Hay datos faltantes en la columna `canal`. 

El dataset tiene 4 columnas tipo 'object' lo que significa que tienen tipos de datos string, como letras, numeros o simbolos y 1 columna tipo 'float' lo que indica que son numeros reales.

Las columnas tipo object son: fecha, pais, id_campaña, canal. La columna tipo float es gasto. 



In [10]:
print('\nValores Faltantes (por columna):')
display(marketing.isna().sum().sort_values(ascending=False).head(15))


Valores Faltantes (por columna):


canal         101
fecha           0
pais            0
id_campaña      0
gasto           0
dtype: int64

✍️ **Comentarios:** 

La siguiente tabla estadistica describe:  

La columna `gasto` tiene 1,620 registros, siendo el valor minimo 501.11 y el valor maximo 2,999.36. Ademas, tiene una desviacion estandar de 734.43 y la media es 1,772.74. 

In [11]:
marketing.describe()

,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


---

### Revisión y calidad de datos 

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets

A. Validar y convertir fechas al formato correcto  
B. Revisar variables numéricas (sin negativos o ceros inválidos)  
C. Verificar consistencia de montos  
D. Eliminar duplicados  
E. Revisar variables categóricas 

*************************************
**A. Validar y convertir fechas al formato correcto:**

✍️ **Comentarios:** 

En el dataset **orders** la columna **fecha_hora_pedido** es tipo 'object'. Esta debe ser modificada a tipo datetime.

In [12]:

orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
orders['fecha_hora_pedido'].isna().sum()


0

In [13]:

# Creacion de columnas separadas de Year, Month and Day. 

orders['year'] = orders['fecha_hora_pedido'].dt.year
orders['month'] = orders['fecha_hora_pedido'].dt.month
orders['day'] = orders['fecha_hora_pedido'].dt.day


In [14]:
# Validacion de la data:

orders.info()
orders.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
 12  year                25100 non-null  int64         
 13  month               25100 non-null  int64     

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,year,month,day
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,2025,5,22
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,2025,6,15
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,2025,5,2
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,2025,6,9
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,2025,3,30


**Validación para el dataset “Marketing”:**

In [15]:
# Validacion fecha dataset "Marketing"
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')
marketing['fecha'].isna().sum()

0

In [16]:
# Creacion de columnas separadas de Year, Month and Day. 

marketing['year'] = marketing['fecha'].dt.year
marketing['month'] = marketing['fecha'].dt.month
marketing['day'] = marketing['fecha'].dt.day

In [17]:
# Validacion de la data:

marketing.info()
marketing.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1519 non-null   object        
 4   gasto       1620 non-null   float64       
 5   year        1620 non-null   int64         
 6   month       1620 non-null   int64         
 7   day         1620 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(3), object(3)
memory usage: 101.4+ KB


,fecha,pais,id_campaña,canal,gasto,year,month,day
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25,2025,1,1
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34,2025,1,1
2,2025-01-01,Mexico,social_Mexico,social,2045.01,2025,1,1
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21,2025,1,1
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40,2025,1,1


✍️ **Comentarios:** 

La descripcion anterior muestra que el formato de la columna `fecha_hora_pais` del dataset "orders" y `fecha` del dataset "marketing" ha sido modificado a tipo **datetime**. 

Ademas, se agregaron tres columnas, separando el ano, mes y dia. 


*****************************
**B. Revisar variables numéricas (sin negativos o ceros inválidos)**

✍️ **Comentarios:** 

Se realiza una validacion de los datos numericos para los tres datasets: `orders`, `catalog` y `marketing`.

Variables a analizar en **`orders`**: cantidad, precio_unitario, monto_descuento, monto_total. 

Variables a analizar en **`catalog`**: costo_unitario.

Variables a analizar en **'`marketing`**: gasto.



**Validacion de la columna `orders`:**

Las columnas `cantidad` y `monto_total` tienen valores negativos y menorers que cero. 

In [18]:
orders[orders['cantidad'] <=0]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,year,month,day
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62,2025,3,13
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50,2025,5,7
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65,2025,2,19
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53,2025,5,25


✍️ **Comentarios:** 

Para corregir los valores negativos de la columna `cantidad`los valores seran reemplazados con valores validos usando la media, de la siguiete forma:

In [19]:
media_cantidad = orders.loc[orders['cantidad'] > 0, 'cantidad'].mean()

In [20]:
orders.loc[orders['cantidad'] <=0, 'cantidad'] = media_cantidad

In [21]:
# Validacion de la columna cantidad usando describe: 
orders['cantidad'].describe()

count    25050.000000
mean         7.094067
std        296.276984
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max      20000.000000
Name: cantidad, dtype: float64

✍️ **Comentarios:** 

La validacion anterior indica que ya no hay valores negativos en la columna `cantidad`. El valor minimo es 1.

**Validacion de la columna `precio_unitario`:**

In [22]:
orders[orders['precio_unitario'] <=0]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,year,month,day


✍️ **Comentarios:** 

No hay valores negativos en la columna `precio_unitario`.

**Validacion de la columna `monto_descuento`:**

In [23]:
orders[orders['monto_descuento'] < 0]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,year,month,day


✍️ **Comentarios:** 

No hay valores negativos en la columna `monto_descuento`

**Validacion de la columna `monto_total`:**

In [24]:
orders[orders['monto_total'] <=0]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,year,month,day
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,7.094067,101.31,10.0,-192.62,2025,3,13
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,7.094067,43.50,5.0,-38.50,2025,5,7
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,7.094067,497.65,5.0,-492.65,2025,2,19
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,7.094067,423.53,0.0,-423.53,2025,5,25


✍️ **Comentarios:** 

La columna `monto_total` tiene valores negativos. Estos seran reemplazados con valores validos usando la media:

In [25]:
media_monto_total = orders.loc[orders['monto_total'] > 0, 'monto_total'].mean()

In [26]:
orders.loc[orders['monto_total'] <= 0, 'monto_total'] = media_monto_total 

In [27]:
# Validacion 
orders[orders['monto_total'] <=0]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,year,month,day


In [28]:
orders['monto_total'].describe()

count    2.510000e+04
mean     2.073056e+03
std      9.894995e+04
min      5.240000e+00
25%      1.806075e+02
50%      3.419250e+02
75%      5.186550e+02
max      8.840200e+06
Name: monto_total, dtype: float64

✍️ **Comentarios:** 

La validacion anterior indica que la columna `monto_total` ya no tiene valores negativos, su valor minimo es 5.24.

**Validacion dateset `catalog`:**

In [29]:
catalog[catalog['costo_unitario'] <= 0]

,nombre_producto,categoria_producto,costo_unitario,proveedor


✍️ **Comentarios:** 

No hay variables numéricas negativas o ceros inválidos en el dataset `catalog`: columna `costo_unitario`.

**Validacion dataset `marketing`:**

In [30]:
marketing[marketing['gasto'] <= 0]

,fecha,pais,id_campaña,canal,gasto,year,month,day


✍️ **Comentarios:** 

No hay variables numéricas negativas o ceros inválidos en el dataset `marketing`, columna `gasto`.

**C. Verificar consistencia de montos**

✍️ **Comentarios:** 

En este paso se hara la validacion  de consistencia de montos y se hara con la formula `total_calculado = cantidad x precio_unitario - monto_descuento`

In [31]:
orders['total_calculado'] = orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento']

In [32]:
inconsistencias = orders[abs(orders['monto_total'] - orders['total_calculado']) > 0.01]

inconsistencias.shape

(1150, 16)

In [33]:
len(inconsistencias)

1150

In [34]:
# Validacion de inconsitencias:

inconsistencias.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,year,month,day,total_calculado
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,2025,5,2,195.98
24,order_24,user_458,2025-01-05,Colombia,NaN,organic,Sneakers-Urban-42,Moda,2.0,117.99,0.0,235.99,2025,1,5,235.98
35,order_35,user_3817,2025-02-08,Mexico,NaN,organic,Blender-XL-Red,Hogar,2.0,467.34,5.0,929.69,2025,2,8,929.68
41,order_41,user_2963,2025-01-27,Mexico,NaN,social,Jacket-Winter-M,Moda,2.0,37.77,10.0,65.53,2025,1,27,65.54
136,order_136,user_6937,2025-02-10,NaN,desktop,organic,Blender-XL-Red,Hogar,2.0,211.05,15.0,407.09,2025,2,10,407.10


✍️ **Comentarios:** 

La tabla anterior muestra la creacion de una columna adicional llamada `total_calculado` en donde se valida que los valores estan relacionados, correctamente calculados y su operacion es valida.

In [35]:
orders['monto_descuento'] = orders['monto_descuento'].fillna(0)

orders['total_calculado'] = (orders['cantidad'] * orders['precio_unitario'] - orders['monto_descuento'])

In [36]:
orders[orders['monto_descuento'] > orders['cantidad'] * orders['precio_unitario']]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,year,month,day,total_calculado


**D. Eliminar duplicados**

Validacion Dataset **orders**:

In [37]:
orders.duplicated().sum()

100

✍️ **Comentarios:** 

El codigo anterior indica que hay 100 filas duplicadas en el dataset **orders** y son los siguientes registros:

In [38]:
orders[orders.duplicated()]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,year,month,day,total_calculado
25000,order_22936,user_4028,2025-06-30,Argentina,desktop,social,Vacuum-Pro-Black,Hogar,2.0,41.91,10.0,73.82,2025,6,30,73.82
25001,order_13710,user_4466,2025-03-20,Mexico,desktop,social,Jacket-Winter-M,Moda,2.0,483.32,0.0,966.65,2025,3,20,966.64
25002,order_14562,user_6590,2025-03-23,Mexico,mobile,paid_search,Tablet-Standard-64GB,Electronica,1.0,481.96,5.0,476.96,2025,3,23,476.96
25003,order_11537,user_3115,2025-03-08,Argentina,desktop,organic,Vacuum-Pro-Black,Hogar,1.0,147.20,5.0,142.20,2025,3,8,142.20
25004,order_4533,user_7944,2025-02-16,Argentina,mobile,organic,Blender-XL-Red,Hogar,1.0,176.91,5.0,171.91,2025,2,16,171.91
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25095,order_3913,user_380,2025-02-18,Argentina,desktop,paid_search,Phone-Pro-128GB,Electronica,1.0,82.28,0.0,82.28,2025,2,18,82.28
25096,order_23405,user_7833,2025-04-04,Colombia,mobile,paid_search,Phone-Pro-128GB,Electronica,2.0,99.25,5.0,193.50,2025,4,4,193.50
25097,order_5615,user_5417,2025-05-13,Colombia,desktop,social,Blender-XL-Red,Hogar,2.0,450.35,5.0,895.69,2025,5,13,895.70
25098,order_812,user_1530,2025-03-31,Argentina,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,167.32,10.0,157.32,2025,3,31,157.32


In [39]:
orders = orders.drop_duplicates()

Validacion de los datos:

In [40]:
orders.duplicated().sum()

0

✍️ **Comentarios:** 

Una vez realizada la limpiea, ya no hay valores duplicados en el dataset **orders**

**Validacion duplicados dataset catalog:**

In [41]:
catalog.duplicated().sum()

0

✍️ **Comentarios:** 

No hay valores duplicados en el dataset **catalog**

**Validacion duplicaods dataset maketing:**

In [42]:
marketing.duplicated().sum()

0

✍️ **Comentarios:** 

No hay valores duplicados en el dataset **marketing**

**E. Revisar variables categóricas**

**Validacion Dataset "orders":**

In [43]:
# Explorando columnas categoricas de "orders": 
columnas_orders = ['pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto']

In [44]:
orders.select_dtypes(include=['object']).columns.tolist()

['id_pedido',
 'id_usuario',
 'pais',
 'dispositivo',
 'fuente_referencia',
 'nombre_producto',
 'categoria_producto']

In [45]:
orders['pais'].unique()

array(['Argentina', 'Mexico', 'Colombia', 'mexico', 'colombia',
       'argentina', nan], dtype=object)

In [46]:
orders['dispositivo'].unique()

array(['desktop', 'mobile', nan], dtype=object)

In [47]:
orders['fuente_referencia'].unique()

array(['organic', 'paid_search', 'social', nan], dtype=object)

In [48]:
orders['nombre_producto'].unique()

array(['Jacket-Winter-M', 'Tablet-Standard-64GB', 'Blender-XL-Red',
       'Laptop-Gaming-16GB', 'Sneakers-Urban-42', 'Phone-Pro-128GB',
       'Vacuum-Pro-Black', nan], dtype=object)

In [49]:
orders['categoria_producto'].unique()

array(['Moda', 'Electronica', 'Hogar', nan], dtype=object)

In [50]:
col_corregida = ['pais']

✍️ **Comentarios:** 

Segun la revision anterior al dataset **orders**, la columna `pais` tiene variables que deben ser corregidas para tener igualdad en el nombre de las categorias. Para realizar lo anterior, se ejecuta el siguiente codigo para eliminar espacios, reemplazar argentina por Argentina, mexico por Mexico y colombia por Colombia. Los valores nulos (nan) NO seran eliminados, seran reemplazados por la palabra 'unknown':

In [51]:
for col in col_corregida:
    orders[col] = orders[col].str.strip()
    orders[col] = orders[col].replace('argentina', 'Argentina')
    orders[col] = orders[col].replace('mexico', 'Mexico')
    orders[col] = orders[col].replace('colombia', 'Colombia')
    orders[col] = orders[col].replace ('nan', 'unknown')

In [52]:

# Validacion de la columna Pais:
orders['pais'].unique()

array(['Argentina', 'Mexico', 'Colombia', nan], dtype=object)

**Validar si hay valores nulos en las columnas de el dataset orders:**

In [53]:
orders[col_corregida].isna().sum()

pais    300
dtype: int64

In [54]:
orders['pais'].unique()

array(['Argentina', 'Mexico', 'Colombia', nan], dtype=object)

In [63]:
# Reemplanzando 'nan' por la palabra 'unknown'

orders[columnas_orders] = orders[columnas_orders].replace ('nan', 'unknown', regex = True)
orders[columnas_orders] = orders[columnas_orders].fillna('unknown')

**Validacion Dataset "catalog":**

In [64]:
# Explorando columnas categoricas de "Catalog"

columnas_catalog = ['nombre_producto', 'categoria_producto', 'proveedor']

In [65]:
catalog.select_dtypes(include=['object']).columns.tolist()

['nombre_producto', 'categoria_producto', 'proveedor']

In [66]:
catalog['nombre_producto'].unique()

array(['Laptop-Gaming-16GB', 'Phone-Pro-128GB', 'Tablet-Standard-64GB',
       'Blender-XL-Red', 'Vacuum-Pro-Black', 'Sneakers-Urban-42',
       'Jacket-Winter-M'], dtype=object)

In [67]:
catalog['categoria_producto'].unique()

array(['Electrónica', 'Hogar', 'Moda'], dtype=object)

In [68]:
catalog['proveedor'].unique()

array(['Fuller, Pena and Myers', 'King Ltd', 'Bowers LLC', 'Long-Reid',
       'Rivera, Carr and Finley', 'Greene-Smith', 'Mcmillan-Rhodes'],
      dtype=object)

✍️ **Comentarios:** 

Segun la validacion anterior, no hay variables a corregir en el dataset **catalog**. 

**Validacion Dataset "Maketing"**

In [69]:
# Explorando columnas categoricas de "Marketing"

columnas_marketing = ['pais', 'canal', 'id_campaña']

In [70]:
marketing.select_dtypes(include=['object']).columns.tolist()

['pais', 'id_campaña', 'canal']

In [71]:
marketing['pais'].unique()

array(['Mexico', 'Colombia', 'Argentina'], dtype=object)

In [72]:
marketing['canal'].unique()

array(['organic', 'paid_search', 'social', nan], dtype=object)

In [73]:
marketing['id_campaña'].unique()

array(['organic_Mexico', 'paid_search_Mexico', 'social_Mexico',
       'organic_Colombia', 'paid_search_Colombia', 'social_Colombia',
       'organic_Argentina', 'paid_search_Argentina', 'social_Argentina'],
      dtype=object)

✍️ **Comentarios:** 

Segun esta validacionel dataset **Marketing** tiene valores "nan" en la columna `canal`, estos valores (nan) NO seran eliminados, seran reemplazados por la palabra 'unknown':

In [74]:
# Reemplanzando 'nan' por la palabra 'unknown'

marketing[columnas_marketing] = marketing[columnas_marketing].replace ('nan', 'unknown', regex = True)
marketing[columnas_marketing] = marketing[columnas_marketing].fillna('unknown')

In [75]:
marketing['canal'].unique()

array(['organic', 'paid_search', 'social', 'unknown'], dtype=object)


**📦 Exportación:** Una vez finalizada la limpieza, se exportan los datasets para ser utilizarlos en la última etapa del proyecto.

In [73]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  


---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

✍️ **Comentarios:** 

Antes de calcular los indicadores clave del negocio, se realizara la union de la dataset **orders** con **catalog** por medio de "merge". 

In [76]:
# Validacion de la columna 'nombre producto' en datasets: "order" y "catalog": 

orders['nombre_producto'].nunique()
catalog['nombre_producto'].nunique()

7

In [77]:
# Se hace la union de las dos datasets: 

orders_catalog = orders.merge(catalog,
                             on = 'nombre_producto',
                             how = 'left')

In [78]:
# La siguiente tabla muestra que las columnas del dataset "catalog" son agregadas en la parte derecha de la tabla: (columnas: categoria_producto, costo_unitario y proveedor)  

orders_catalog.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto_x,cantidad,precio_unitario,monto_descuento,monto_total,year,month,day,total_calculado,categoria_producto_y,costo_unitario,proveedor
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,2025,5,22,665.38,Moda,189.31,Mcmillan-Rhodes
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,2025,6,15,171.86,Electrónica,25.21,Bowers LLC
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,2025,5,2,195.98,Hogar,176.64,Long-Reid
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,2025,6,9,242.87,Electrónica,25.21,Bowers LLC
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,2025,3,30,336.28,Hogar,176.64,Long-Reid


In [80]:
# Validacion de la columna "costo_unitario":

orders_catalog['costo_unitario'].isna().sum()

30

✍️ **Comentarios:** 

El resultado anterior indica que hay 30 productos sin costo.

Teniendo en cuenta que el dataset es de 25.100 registros, y los datos faltantes son 30 (minoria), estos productos sin costo seran eliminados:

In [81]:
orders_catalog = orders_catalog.dropna(subset = ['costo_unitario'])

In [82]:
# Validacion nuevamente de la columna "costo_unitario" despues de hacer la eliminacion:

orders_catalog['costo_unitario'].isna().sum()

0

In [83]:
# Creacion columna "Costo Total"

orders_catalog['costo_total'] = (orders_catalog['cantidad'] * orders_catalog['costo_unitario'])

In [84]:
# Creacion de la columna "Profit" (Ganancia)

orders_catalog['profit'] = (orders_catalog['monto_total'] - orders_catalog['costo_total'])

In [85]:
orders_catalog.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto_x,cantidad,precio_unitario,...,monto_total,year,month,day,total_calculado,categoria_producto_y,costo_unitario,proveedor,costo_total,profit
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,...,665.37,2025,5,22,665.38,Moda,189.31,Mcmillan-Rhodes,378.62,286.75
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,...,171.86,2025,6,15,171.86,Electrónica,25.21,Bowers LLC,25.21,146.65
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,...,195.99,2025,5,2,195.98,Hogar,176.64,Long-Reid,353.28,-157.29
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,...,242.87,2025,6,9,242.87,Electrónica,25.21,Bowers LLC,25.21,217.66
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,...,336.28,2025,3,30,336.28,Hogar,176.64,Long-Reid,176.64,159.64


**📊 Parte 1: Rentabilidad del negocio**

- **Ingreso total (revenue):**  (Suma de 'monto_total' en orders)

In [86]:
revenue_total = orders_catalog['monto_total'].sum()

print("Revenue Total:", revenue_total)

Revenue Total: 51984639.16264265


✍️ **Comentarios:** 

El Revenue Total o Ingresos por ventas generados por RappiPlus es de $51,984,639.16

- **Costo total de productos:** 

In [87]:
orders_catalog['costo_total'] = (orders_catalog['cantidad'] * orders_catalog['costo_unitario'])

In [88]:
costo_total = orders_catalog['costo_total'].sum()

print("Costo Total:", costo_total)

Costo Total: 43124356.177828796


✍️ **Comentarios:** 

El costo total generados por RappiPlus es de $43,124,356.17

- **Inversion en marketing:** (suma del gasto en marketing) 

In [89]:
inversion_marketing = marketing['gasto'].sum()

print("Inversion en Marketing:", inversion_marketing)

Inversion en Marketing: 2871843.53


✍️ **Comentarios:** 

El total gastado en campanas de Marketing ha sido de $2,871,843.53

- **Ganancia bruta:** (Revenue - Costo de Productos)

In [90]:
ganancia_bruta = revenue_total - costo_total

print("Ganancia Bruta:", ganancia_bruta)

Ganancia Bruta: 8860282.984813854


✍️ **Comentarios:** 

La ganancia bruta es de $8,860,282.99

In [92]:
# Validacion de valores

orders_catalog[['precio_unitario', 'costo_unitario']].describe()

,precio_unitario,costo_unitario
count,24920.000000,24970.000000
mean,259.368268,102.035757
std,138.682287,98.787488
min,20.030000,10.120000
25%,138.517500,16.600000
50%,258.775000,25.210000
75%,380.372500,189.310000
max,499.960000,280.680000


✍️ **Comentarios:** 

La validacion anterior muestra que no hay perdidas. 

- **Ganancia neta:** (Revenue - costo de productos - marketing) 

In [93]:
ganancia_neta = revenue_total - costo_total - inversion_marketing

print("Ganancia Neta:", ganancia_neta)

Ganancia Neta: 5988439.454813855


✍️ **Comentarios:** 

La ganancia neta muestra un valor positivo de $5,988,439.45. Esto representa que el Proyecto RappiPlus es rentable. Esta ganancia se genera despues de pagar paroductos y marketing. 

- **Margen de Ganancia:** Ganancia neta / Revenue x 100

In [94]:
margen_ganancia = (ganancia_neta / revenue_total) * 100

print("Margen de Ganancia: %", margen_ganancia)

Margen de Ganancia: % 11.519632628550173


El margen de ganancia es de 11.52%

- **ROI de Marketing:**

In [95]:
roi = (ganancia_neta / inversion_marketing) * 100

print("ROI:", roi)

ROI: 208.5224836331475


In [99]:

# Por cada $1 invertido en marketing, el Proyecto RappiPlus genera $2 de ganancia.


**Parte 2: Comportamiento de ventas**

- **Ticket promedio por orden:** 

In [100]:
ticket_promedio = (orders_catalog.groupby('id_pedido')['monto_total'].sum().mean())

print("Ticket Promedio:", ticket_promedio)

Ticket Promedio: 2081.8838270982237


El valor promedio por cada orden es de $2,081.88

- **Cantidad promedio de productos por orden:** 

In [101]:
cantidad_promedio = (orders_catalog.groupby('id_pedido')['cantidad'].sum().mean())

print("Cantidad promedio:", cantidad_promedio)

Cantidad promedio: 7.108945785649478


La cantidad promedio de productos por orden es de: 7.10

- **Producto más vendido:**

In [102]:
producto_mas_vendido = (orders_catalog.groupby('nombre_producto')['cantidad']
                       .sum()
                       .sort_values(ascending=False)
                       .head(1)
                       )

print("Producto mas vendido:", producto_mas_vendido)

Producto mas vendido: nombre_producto
Laptop-Gaming-16GB    144198.0
Name: cantidad, dtype: float64


El producto con mayor volumen de ventas es **Laptop-Gaming-16GB** con una cantidad de 144,198. durante el periodo 2025.

- **Gasto en marketing por canal** 

In [103]:
gasto_por_canal = (
    marketing.groupby('canal')['gasto']
    .sum()
    .sort_values(ascending=False)
)

print("Gasto por canal:", gasto_por_canal)

Gasto por canal: canal
social         918043.21
organic        913533.01
paid_search    863088.21
unknown        177179.10
Name: gasto, dtype: float64


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**

- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario
  


---


**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?


---

In [104]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

**📊 Parte 1: Construcción del funnel**

- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario

In [105]:

# Explorar tabla events
# =========================

query_events = '''
SELECT *
FROM events
LIMIT 10;
'''
events = pd.read_sql(query_events, con=engine)
events.head()



,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [106]:
# Explorando nombre del evento y usuarios: 
query_events = '''
SELECT nombre_evento,
       COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY nombre_evento
ORDER BY usuarios DESC;
'''
events = pd.read_sql(query_events, con=engine)
events.head(10)


,nombre_evento,usuarios
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [107]:
# Explorando usuarios por pais: 

query_events = '''

SELECT pais,
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY pais
ORDER BY usuarios DESC;

'''
events = pd.read_sql(query_events, con=engine)
events.head()

,pais,usuarios
0,Mexico,2697
1,Colombia,2688
2,Argentina,2615


In [108]:
# Explorando dispositivos por usuarios: 

query_events = '''
SELECT dispositivo,
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY dispositivo
ORDER BY usuarios DESC;

'''
events = pd.read_sql(query_events, con=engine)
events.head()


,dispositivo,usuarios
0,desktop,4058
1,mobile,3942


In [109]:
# Explorando la variable fuente_referencia por usuarios: 

query_events = '''
SELECT fuente_referencia, 
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY fuente_referencia
ORDER BY usuarios DESC;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,fuente_referencia,usuarios
0,social,7946
1,organic,7945
2,paid_search,7943


In [110]:
# Explorando la variable categoria_producto por usuarios: 

query_events = '''
SELECT categoria_producto, 
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY categoria_producto
ORDER BY usuarios DESC;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,categoria_producto,usuarios
0,Electronica,7959
1,Moda,7948
2,Hogar,7943


In [111]:
query_events = '''
SELECT 
    COUNT(DISTINCT CASE
        WHEN nombre_evento = 'first_visit'
        THEN id_usuario
    END) AS usuarios_visita,

    COUNT(DISTINCT CASE
        WHEN nombre_evento = 'add_to_cart'
        THEN id_usuario
    END) AS agrega_carrito,

    COUNT(DISTINCT CASE
        WHEN nombre_evento = 'select_item'
        THEN id_usuario
    END) AS seleciona_item, 

    COUNT(DISTINCT CASE
        WHEN nombre_evento = 'begin_checkout'
        THEN id_usuario
    END) AS inicia_compra,

    COUNT(DISTINCT CASE
        WHEN nombre_evento = 'add_payment_info'
        THEN id_usuario
    END) AS agrega_info_pago, 

    COUNT(DISTINCT CASE
        WHEN nombre_evento = 'purchase'
        THEN id_usuario
    END) AS finaliza_compra
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,usuarios_visita,agrega_carrito,seleciona_item,inicia_compra,agrega_info_pago,finaliza_compra
0,7796,7634,7582,7208,6250,6240


**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?

- **Se calcula la tasa de conversión entre cada paso del funnel**

In [112]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios,
    CASE
        WHEN nombre_evento = 'first_visit' THEN 1
        WHEN nombre_evento = 'add_to_cart' THEN 2
        WHEN nombre_evento = 'select_item' THEN 3
        WHEN nombre_evento = 'begin_checkout' THEN 4
        WHEN nombre_evento = 'add_payment_info' THEN 5
        WHEN nombre_evento = 'purchase' THEN 6
    END AS orden
FROM events
WHERE nombre_evento IN(
    'first_visit', 
    'add_to_cart', 
    'select_item', 
    'begin_checkout', 
    'add_payment_info', 
    'purchase'
)
GROUP BY nombre_evento
)

SELECT
    nombre_evento, 
    usuarios, 
    LAG(usuarios) OVER(ORDER BY orden) AS usuarios_etapa_anterior, 

    ROUND(
    usuarios * 100.0 /
    LAG(usuarios) OVER (ORDER BY orden),
    2)
    AS tasa_conversion
FROM funnel
ORDER BY orden;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios,usuarios_etapa_anterior,tasa_conversion
0,first_visit,7796,NaN,NaN
1,add_to_cart,7634,7796.0,97.92
2,select_item,7582,7634.0,99.32
3,begin_checkout,7208,7582.0,95.07
4,add_payment_info,6250,7208.0,86.71
5,purchase,6240,6250.0,99.84


✍️ **Comentarios:** 

La tabal anterior indica que: 

  el 97.92% de los usuarios que visitaron la pagina agregaron un producto al carrito.

  el 95.07% de los usuarios que seleccionaron un item iniciaron el proceso de pago.
  
  el 99.84% de los usuarios que agregaron la informacion de pago terminaron con exito el proceso de compra del producto. 

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3
     
Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [113]:

# Explorando la tabla users
# =========================

query_events = '''
SELECT *
FROM users
LIMIT 10;
'''
events = pd.read_sql(query_events, con=engine)
events.head()



,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free
3,user_3,2025-03-04,Mexico,desktop,free
4,user_4,2025-02-27,Argentina,desktop,free


In [114]:

# Explorando la tabla user_activity
# =========================

query_events = '''
SELECT *
FROM user_activity
LIMIT 10;
'''
events = pd.read_sql(query_events, con=engine)
events.head()



,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0


**1. Se identifica la cohorte de cada usuario según el **mes de registro**.**

In [115]:
# Validando formato de la fecha
# =========================
query_users = '''
SELECT id_usuario, 
    CAST(fecha_registro AS DATE) as fecha_registro,
    DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
FROM users
ORDER BY fecha_registro;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)


,id_usuario,fecha_registro,cohorte
0,user_5408,2025-01-01,2025-01-01 00:00:00+00:00
1,user_145,2025-01-01,2025-01-01 00:00:00+00:00
2,user_6452,2025-01-01,2025-01-01 00:00:00+00:00


In [116]:
# =========================
query_users = '''
WITH usuarios AS (
    SELECT id_usuario,
    CAST(fecha_registro AS DATE) AS fecha_registro
FROM users
), 

actividad AS (
    SELECT id_usuario,
    CAST(fecha_actividad AS DATE) AS fecha_actividad
FROM user_activity
WHERE activo = 1
)

SELECT 
    u.id_usuario,
    u.fecha_registro,
    DATE_TRUNC('month', u.fecha_registro)::DATE AS cohorte,
    a.fecha_actividad,

    FLOOR(
    (a.fecha_actividad - u.fecha_registro) / 7.0
    ) AS semanas_desde_registro

FROM usuarios AS u

INNER JOIN actividad AS a
    ON u.id_usuario = a.id_usuario

WHERE a.fecha_actividad >= u.fecha_registro

ORDER BY
    cohorte,
    u.id_usuario,
    semanas_desde_registro;


'''
users = pd.read_sql(query_users, con=engine)
users.head(5)


,id_usuario,fecha_registro,cohorte,fecha_actividad,semanas_desde_registro
0,user_0,2025-01-29,2025-01-01,2025-02-12,2.0
1,user_0,2025-01-29,2025-01-01,2025-02-19,3.0
2,user_1,2025-01-07,2025-01-01,2025-01-28,3.0
3,user_10,2025-01-09,2025-01-01,2025-01-16,1.0
4,user_10,2025-01-09,2025-01-01,2025-01-23,2.0


********************

**2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.**
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


In [117]:

#
# =========================
query_user_activity = '''



WITH usuarios AS (
    SELECT 
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_registro
FROM users
), 



actividad AS (
    SELECT 
        id_usuario,
        CAST(fecha_actividad AS DATE) AS fecha_actividad
FROM user_activity
WHERE activo = 1
),




base_retencion AS(
    SELECT u.id_usuario,
    DATE_TRUNC('month', u.fecha_registro)::DATE AS cohorte,
    
    FLOOR(
    (a.fecha_actividad - u.fecha_registro) / 7.0
    ) AS semanas_desde_registro


FROM usuarios AS u





INNER JOIN actividad AS a
    ON u.id_usuario = a.id_usuario

WHERE a.fecha_actividad >= u.fecha_registro
),

retencion_por_usuario AS (
    SELECT 
        id_usuario, 
        cohorte, 


        
        MAX(CASE WHEN semanas_desde_registro = 1 THEN 1
            ELSE 0 
            END) AS retenido_w1,

        MAX(CASE WHEN semanas_desde_registro = 2 THEN 1
            ELSE 0
            END) AS retenido_w2, 

        MAX(CASE WHEN semanas_desde_registro = 3 THEN 1
            ELSE 0
            END) AS retenido_w3

FROM base_retencion

GROUP BY
    id_usuario,
    cohorte
    )



SELECT 
    cohorte, 

    COUNT(DISTINCT id_usuario) AS usuarios_con_actividad,

    SUM(retenido_w1) AS retenidos_w1, 
    SUM(retenido_w2) AS retenidos_w2,
    SUM(retenido_w3) AS retenidos_w3

FROM retencion_por_usuario

GROUP BY cohorte

ORDER BY cohorte;

'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(5)


,cohorte,usuarios_con_actividad,retenidos_w1,retenidos_w2,retenidos_w3
0,2025-01-01,1381,697,668,656
1,2025-02-01,1255,611,609,635
2,2025-03-01,1428,677,705,690
3,2025-04-01,1394,680,697,663
4,2025-05-01,1446,695,676,706


In [118]:
#
# =========================
query_user_activity = '''

SELECT COUNT(*) AS total_usuarios
FROM users;

'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,total_usuarios
0,8000


**3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:**

* semana_1: porcentaje de usuarios retenidos en la semana 1

* semana_2: porcentaje de usuarios retenidos en la semana 2

* semana_3: porcentaje de usuarios retenidos en la semana 3


In [119]:
# 
# ======================

query_cohort_retention_final = '''

SELECT 
    DATE_TRUNC('month', CAST(u.fecha_registro AS DATE))::DATE AS cohorte,

    COUNT(DISTINCT u.id_usuario) AS usuarios_registrados, 

    ROUND(
        COUNT(DISTINCT CASE
        WHEN ua.activo = 1
        AND ua.dias_despues_registro BETWEEN 7 AND 13
        THEN u.id_usuario
        END) * 100.0
        / COUNT(DISTINCT u.id_usuario), 2) AS semana_1, 

    ROUND(
        COUNT(DISTINCT CASE
        WHEN ua.activo = 1
        AND ua.dias_despues_registro BETWEEN 14 AND 20
        THEN u.id_usuario
        END) * 100.0
        / COUNT(DISTINCT u.id_usuario), 2) AS semana_2, 

    ROUND(
        COUNT(DISTINCT CASE
        WHEN ua.activo = 1
        AND ua.dias_despues_registro BETWEEN 21 AND 27
        THEN u.id_usuario
        END) * 100.0
        / COUNT(DISTINCT u.id_usuario), 2) AS semana_3

FROM users u

LEFT JOIN user_activity ua
    ON u.id_usuario = ua.id_usuario

GROUP BY cohorte

ORDER BY cohorte;

'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_registrados,semana_1,semana_2,semana_3
0,2025-01-01,1627,42.84,41.06,40.32
1,2025-02-01,1444,42.31,42.17,43.98
2,2025-03-01,1636,41.38,43.09,42.18
3,2025-04-01,1606,42.34,43.40,41.28
4,2025-05-01,1687,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.
2. **Plantear la hipótesis estadística**
3. **Aplicar el test estadístico adecuado**
4. **Interpretar el resultado**  

In [120]:


import numpy as np
import pandas as pd

# Para visualizaciones: 
import matplotlib.pyplot as plt
import seaborn as sns

# Para estadisticas y correlaciones: 
from scipy.stats import pointbiserialr
from scipy.stats import chi2_contingency
from scipy import stats
from IPython.display import display 
from statsmodels.stats.proportion import proportions_ztest

In [121]:
# cargar archivo

experiment_checkout = pd.read_csv('experiment_checkout_ui.csv')

**1. Analizando el dataset para identificar la metrica principal:** 

In [122]:
experiment_checkout.info(5)
experiment_checkout.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_usuario       10000 non-null  object 
 1   variante         10000 non-null  object 
 2   convirtio        10000 non-null  int64  
 3   dispositivo      10000 non-null  object 
 4   pais             10000 non-null  object 
 5   duracion_sesion  10000 non-null  float64
 6   timestamp        10000 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 547.0+ KB


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [41]:
print('\nValores Faltantes (por columna):')
display(experiment_checkout.isna().sum().sort_values(ascending=False).head(15))


Valores Faltantes (por columna):


id_usuario         0
variante           0
convirtio          0
dispositivo        0
pais               0
duracion_sesion    0
timestamp          0
dtype: int64

✍️ **Comentarios:** 

* Hay un total de 7 columnas con 10,000 registros para cada columna. El dataset no tiene valores ausentes.

*  Las siguientes 5 columnas son tipo **objetc:** `id_usuario`, `variante`, `dispositivo`, `pais`, `timestamp`.

*  La columna `convirtio` es tipo **int** y la columna `duracion_sesion` es tipo **float**.

*  La columna `timestamp` es tipo **object**. Esta debe ser modificada a tipo datetime.

*  Memory usage: 547.0+ KM

In [123]:
experiment_checkout['timestamp'] = pd.to_datetime(experiment_checkout['timestamp'], errors='coerce')
experiment_checkout['timestamp'].isna().sum()

0

In [124]:
# Identificando el rango temporal de las variables: 

print('Fecha Mínima:', experiment_checkout['timestamp'].min())
print('Fecha Máxima:', experiment_checkout['timestamp'].max())

Fecha Mínima: 2025-01-01 00:00:00
Fecha Máxima: 2025-06-30 00:00:00


In [125]:
# Creacion de columnas separadas de Year, Month and Day. 

experiment_checkout['year'] = experiment_checkout['timestamp'].dt.year
experiment_checkout['month'] = experiment_checkout['timestamp'].dt.month
experiment_checkout['day'] = experiment_checkout['timestamp'].dt.day

In [127]:
# Validacion de tipo de datp en la columna timestamp:

experiment_checkout.info(5)
experiment_checkout.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_usuario       10000 non-null  object        
 1   variante         10000 non-null  object        
 2   convirtio        10000 non-null  int64         
 3   dispositivo      10000 non-null  object        
 4   pais             10000 non-null  object        
 5   duracion_sesion  10000 non-null  float64       
 6   timestamp        10000 non-null  datetime64[ns]
 7   year             10000 non-null  int64         
 8   month            10000 non-null  int64         
 9   day              10000 non-null  int64         
dtypes: datetime64[ns](1), float64(1), int64(4), object(4)
memory usage: 781.4+ KB


,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp,year,month,day
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28,2025,3,28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15,2025,1,15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18,2025,3,18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03,2025,6,3
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12,2025,1,12


In [128]:
# Validacion de duplicados en el dataset: 

duplicados_experiment_checkout = experiment_checkout['id_usuario'].duplicated().sum()
print('Duplicados en id_usuario:', duplicados_experiment_checkout)

Duplicados en id_usuario: 0



✍️ **Comentario:** 

No hay evidencia de usuarios duplicados. 


In [129]:
# Valores unicos en las variables convirtio

print('Valores en convirtio:', sorted(experiment_checkout['convirtio'].dropna().unique()))

Valores en convirtio: [0, 1]


✍️ **Comentario:** 

La metrica principal es `conversion` donde **1** es el usuario completo la compra y **0** el usuario no completo la compra. 


In [130]:
print('Valores en Variante:', sorted(experiment_checkout['variante'].dropna().unique()))

print("\nConteo de categorías:")
print(experiment_checkout['variante'].value_counts())

Valores en Variante: ['control', 'tratamiento']

Conteo de categorías:
tratamiento    5035
control        4965
Name: variante, dtype: int64


✍️ **Comentario:** 

La categoria **tratamiento** tiene 5,035 valores y la categoria **control** tiene 4,965 

2. **Plantear la hipótesis estadística**  
   
   - **H₀ (Hipótesis nula):** No hay diferencia en conversion entre los grupos. La interfaz modificada no tiene cambios significativos en la tasa de conversion respecto a la interfaz original.
   - **H₁ (Hipótesis alternativa):** Hay diferencia significativa en conversion. La interfaz modificada  incrementa la tasa de conversion respecto a la interfaz original. 

3. **Aplicar el test estadístico adecuado** 

In [131]:
# Resumen por variante tamaño y duracion sesion

resumen_duracion_sesion = experiment_checkout.groupby('variante')['duracion_sesion'].agg(['count', 'mean', 'median', 'std']).round(2)
resumen_duracion_sesion.rename(columns={'count':'n', 'mean':'duracion_sesion_promedio', 'median': 'duracion_sesion_mediana', 'std': 'duracion_sesion_std'}) 

,n,duracion_sesion_promedio,duracion_sesion_mediana,duracion_sesion_std
variante,,,,
control,4965,161.04,161.28,81.47
tratamiento,5035,158.70,158.07,80.68


In [132]:
conversion = (experiment_checkout.groupby('variante')['convirtio'].mean().mul(100).round(2))
print(conversion)

variante
control        15.69
tratamiento    16.29
Name: convirtio, dtype: float64


✍️ **Comentario:**

El grupo **control** tiene 15.69% y el grupo **tratamiento** tiene 16.29% lo que significa que **tratamiento** tiene una tasa de conversión mayor de 0.60%.

In [133]:
# Tabla de contingencia (variante vs convirtio)

tabla = pd.crosstab(experiment_checkout['variante'], experiment_checkout['convirtio'])
print(tabla)

convirtio       0    1
variante              
control      4186  779
tratamiento  4215  820


✍️ **Comentario:**

La tabla anterior muestra que en el grupo **control**, un total de 4,186 usuarios no completaron la compra, mientras que 779 completaron la transacción. 

Por otro lado, en el grupo **tratramiento**, muestra que 4,215 usuarios no compleataron la compra, mientras que 820 si concretaron una compra. 


In [134]:
# Prueba Chi-cuadrado

chi2, p_value, dof, expected = chi2_contingency(tabla)

print('chi2:', round(chi2, 4))
print('p_value:', round(p_value, 6))
print('grados de libertad (dof):', dof)

chi2: 0.6178
p_value: 0.431871
grados de libertad (dof): 1


In [135]:
expected_df = pd.DataFrame(expected, index=tabla.index, columns = tabla.columns).round(2)
expected_df

convirtio,0,1
variante,,
control,4171.1,793.9
tratamiento,4229.9,805.1


In [136]:
alpha = 0.05 # umbral de significancia

In [137]:
print(f"Valor P: {p_value}")

if p_value < alpha: 
    print("Se rechaza Ho, hay diferencia significativa.")
else: 
    print("No se rechaza Ho, no hay evidencia de diferencia.")

Valor P: 0.43187130118526496
No se rechaza Ho, no hay evidencia de diferencia.


4. **Interpretar el resultado**

✍️ **Comentarios:** 

Segun los resultados anteriores, se puede determinar que: 

- El valor de p_value es 0.4319
- El valor de Chi2 es 0.6178
- Nivel de significancia es 5% (alpha = 0.05)

Entonces: 0.4319 > 0.05, lo que indica que  **No** se rechaza la hipotesis nula Ho. No hay evidencia que indique que la nueva interfaz de checkout haya modificado la tasa de conversion respecto a la interfaz original.

Ademas, tenienedo en cuenta que el valor de chi2 es bajo, las frecuencias observadas son muy parecidas a las esperadas. 


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

 📎 **Enlace del Dashboard:**

# Link

https://app.powerbi.com/groups/me/reports/79c66a6c-8460-406b-a057-0051fd0ee9f8/06516da24fd6b39a8ba3?experience=power-bi